<a href="https://colab.research.google.com/github/tazir-shaif/ai-engineering-portfolio/blob/main/module-5-agents/Module_5_Session_5_Multi_Agent_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 5 — Session 5: Multi Agent Systems

## What are we building?
A Swiggy support system with four specialist agents:
- Router Agent — receives query and routes to right specialist
- Order Agent — handles order status and tracking
- Refund Agent — handles refunds and compensation
- Escalation Agent — handles complex complaints

## Why does this matter?
One agent cannot scale to millions of queries. Real production
systems at Swiggy, Zepto, and Flipkart use specialist agents
that each handle one domain extremely well. Routing means faster,
more accurate responses and easier debugging.

## What we will learn:
- How to build specialist agents with focused tools
- How a router agent decides which specialist to call
- How agents hand off work to each other
- Why multi agent systems are more maintainable at scale

## Model: openai/gpt-oss-20b on Groq
## No new libraries this session
## Platform: Google Colab

## Step 1: Install and Setup
Same libraries as previous sessions. No new libraries this session.
We are focusing purely on the multi agent architecture concept.

In [ ]:
# Same setup as previous sessions — no new libraries
!pip install langgraph langchain-groq langchain -q

from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage
from langchain_core.tools import tool
from typing import TypedDict, Annotated, Literal
import operator
import os
import json
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# One LLM — all agents share the same model
# In production different agents could use different models
# e.g. router uses small fast model, refund agent uses larger accurate model
llm = ChatGroq(model="openai/gpt-oss-20b")

print("Setup complete")
print("LLM:", llm.model_name)

## Step 2: Define Specialist Tools
Each specialist agent gets only the tools it needs.
Order Agent gets order tools only.
Refund Agent gets refund tools only.
This is the key difference from previous sessions where
one agent had access to all tools at once.

In [ ]:
# --- ORDER AGENT TOOLS ---
@tool
def check_order_status(order_id: str) -> str:
    """Check the current status of a Swiggy order using the order ID."""
    orders = {
        "ORD001": {"status": "Delayed", "eta": "45 minutes", "restaurant": "Burger King"},
        "ORD002": {"status": "Delivered", "eta": None, "restaurant": "Pizza Hut"},
        "ORD003": {"status": "Preparing", "eta": "25 minutes", "restaurant": "McDonald's"},
    }
    order = orders.get(order_id)
    if order:
        return f"Order {order_id} from {order['restaurant']}: {order['status']}. ETA: {order['eta']}"
    else:
        return f"Order {order_id} not found in system."

@tool
def get_delivery_partner_info(order_id: str) -> str:
    """Get delivery partner information for a Swiggy order."""
    partners = {
        "ORD001": {"name": "Ravi Kumar", "phone": "98XXXXXX01", "location": "2km away"},
        "ORD002": {"name": "Suresh M", "phone": "98XXXXXX02", "location": "Delivered"},
        "ORD003": {"name": "Amit Singh", "phone": "98XXXXXX03", "location": "At restaurant"},
    }
    partner = partners.get(order_id)
    if partner:
        return f"Delivery partner for {order_id}: {partner['name']}, Location: {partner['location']}"
    else:
        return f"No delivery partner assigned for {order_id} yet."

# --- REFUND AGENT TOOLS ---
@tool
def calculate_refund(order_id: str, reason: str) -> str:
    """Calculate refund amount for a Swiggy order.
    Reason must be either wrong_item or late_delivery."""
    order_values = {"ORD001": 450, "ORD002": 320, "ORD003": 180}
    amount = order_values.get(order_id, 0)
    if reason == "wrong_item":
        refund = amount * 1.0
    elif reason == "late_delivery":
        refund = amount * 0.3
    else:
        refund = 0
    return f"Refund for {order_id}: ₹{refund} approved for reason: {reason}"

@tool
def check_refund_status(order_id: str) -> str:
    """Check if a refund has already been processed for an order."""
    refunds = {
        "ORD001": {"status": "Processed", "amount": 135, "date": "2026-06-25"},
        "ORD002": {"status": "Pending", "amount": 320, "date": "2026-06-27"},
    }
    refund = refunds.get(order_id)
    if refund:
        return f"Refund for {order_id}: {refund['status']} — ₹{refund['amount']} on {refund['date']}"
    else:
        return f"No refund record found for {order_id}."

# --- ESCALATION AGENT TOOLS ---
@tool
def create_complaint_ticket(order_id: str, complaint_type: str, description: str) -> str:
    """Create a formal complaint ticket for serious issues."""
    ticket_id = f"TKT{order_id[-3:]}001"
    return f"Complaint ticket {ticket_id} created for {order_id}. Type: {complaint_type}. Team will respond within 24 hours."

@tool
def escalate_to_human(order_id: str, reason: str) -> str:
    """Escalate issue to human support agent for complex problems."""
    return f"Order {order_id} escalated to senior support. Reason: {reason}. Agent will call within 2 hours."

# Print summary
print("Order Agent tools:", ["check_order_status", "get_delivery_partner_info"])
print("Refund Agent tools:", ["calculate_refund", "check_refund_status"])
print("Escalation Agent tools:", ["create_complaint_ticket", "escalate_to_human"])

## Step 3: Build Specialist Agents
Each specialist agent is a mini LangGraph — same pattern as Session 3.
The difference is each agent has its own tools and its own system prompt
that makes it an expert in its domain.
We wrap each agent in a function that takes a query and returns an answer.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

def build_specialist_agent(system_prompt, tools):
    """
    Reusable function to build a specialist agent.
    Takes a system prompt and a list of tools.
    Returns a compiled LangGraph graph.
    DRY principle — one function builds all our specialist agents.
    """
    # Bind tools to LLM
    llm_with_tools = llm.bind_tools(tools)
    tool_executor = {t.name: t for t in tools}

    # Define state
    class AgentState(TypedDict):
        messages: Annotated[list, operator.add]

    # Agent node
    def agent_node(state):
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response]}

    # Tool node
    def tool_node(state):
        last_message = state["messages"][-1]
        results = []
        for tool_call in last_message.tool_calls:
            tool_name = tool_call["name"]
            tool_args = tool_call["args"]
            tool_result = tool_executor[tool_name].invoke(tool_args)
            results.append(ToolMessage(
                content=tool_result,
                tool_call_id=tool_call["id"]
            ))
        return {"messages": results}

    # Router
    def should_continue(state):
        last_message = state["messages"][-1]
        if last_message.tool_calls:
            return "tool_node"
        return END

    # Build graph
    graph_builder = StateGraph(AgentState)
    graph_builder.add_node("agent_node", agent_node)
    graph_builder.add_node("tool_node", tool_node)
    graph_builder.set_entry_point("agent_node")
    graph_builder.add_conditional_edges("agent_node", should_continue, {
        "tool_node": "tool_node",
        END: END
    })
    graph_builder.add_edge("tool_node", "agent_node")

    return graph_builder.compile()

def run_specialist_agent(agent_graph, system_prompt, query):
    """
    Run a specialist agent on a query.
    Returns the final answer as a string.
    """
    result = agent_graph.invoke({
        "messages": [
            SystemMessage(content=system_prompt),
            HumanMessage(content=query)
        ]
    })
    return result["messages"][-1].content

# --- BUILD ALL THREE SPECIALIST AGENTS ---
# Order Agent
order_agent = build_specialist_agent(
    system_prompt="""You are a Swiggy Order Specialist.
    You only handle order status and delivery tracking queries.
    Always check order status first, then provide delivery partner
    info if customer needs more details.""",
    tools=[check_order_status, get_delivery_partner_info]
)

# Refund Agent
refund_agent = build_specialist_agent(
    system_prompt="""You are a Swiggy Refund Specialist.
    You only handle refund and compensation queries.
    Always check if a refund already exists before creating a new one.
    Be empathetic and clear about refund timelines.""",
    tools=[calculate_refund, check_refund_status]
)

# Escalation Agent
escalation_agent = build_specialist_agent(
    system_prompt="""You are a Swiggy Escalation Specialist.
    You handle complex complaints that cannot be resolved by other agents.
    Always create a complaint ticket first, then escalate to human if needed.
    Be professional and reassuring.""",
    tools=[create_complaint_ticket, escalate_to_human]
)

print("Specialist agents built:")
print("  - Order Agent")
print("  - Refund Agent")
print("  - Escalation Agent")

## Step 4: Build the Router Agent
The Router Agent is the brain of our multi agent system.
It receives every customer query first and decides:
- Is this an order tracking question? → Order Agent
- Is this a refund question? → Refund Agent  
- Is this a complex complaint? → Escalation Agent
The router does NOT answer the customer directly.
It only routes to the right specialist.

In [ ]:
# Router uses plain LLM — no tools needed
# It only needs to read the query and decide which agent to call
router_llm = ChatGroq(model="openai/gpt-oss-20b")

ROUTER_PROMPT = """You are a Swiggy customer support router.
Your only job is to read the customer query and decide which
specialist agent should handle it.

Reply with EXACTLY one of these three words — nothing else:
- ORDER — if customer asks about order status, delivery tracking,
  delivery partner, or where their order is
- REFUND — if customer asks about refund, compensation, wrong item,
  late delivery payment, or money back
- ESCALATION — if customer is angry, has a serious complaint,
  mentions rude behaviour, food poisoning, or needs urgent help

Reply with only one word. No explanation."""

def route_query(customer_query):
    """
    Router agent — reads query and returns which specialist to use.
    Returns: "ORDER", "REFUND", or "ESCALATION"
    """
    response = router_llm.invoke([
        SystemMessage(content=ROUTER_PROMPT),
        HumanMessage(content=customer_query)
    ])

    # Clean the response — remove spaces and make uppercase
    decision = response.content.strip().upper()

    # Safety check — if LLM returns something unexpected, default to escalation
    if decision not in ["ORDER", "REFUND", "ESCALATION"]:
        print(f"  Router returned unexpected value: {decision} — defaulting to ESCALATION")
        decision = "ESCALATION"

    return decision

# --- TEST ROUTER ALONE ---
test_queries = [
    "Where is my order ORD001?",
    "I want a refund for wrong item in ORD002",
    "The delivery partner was extremely rude to me"
]

print("Testing router decisions:")
print("-" * 50)
for query in test_queries:
    decision = route_query(query)
    print(f"Query: {query}")
    print(f"Router decision: {decision}")
    print()

## Step 5: Build the Full Multi Agent Pipeline
Now we connect everything together.
Customer query → Router → Specialist Agent → Final Answer
This is the complete multi agent system.

In [ ]:
def run_multi_agent_system(customer_query):
    """
    Full multi agent pipeline:
    1. Router reads query and decides which specialist to call
    2. Specialist agent handles the query with its own tools
    3. Final answer returned to customer
    """
    print(f"Customer: {customer_query}")
    print("=" * 60)

    # --- STEP 1: Router decides which specialist to call ---
    decision = route_query(customer_query)
    print(f"Router decision: {decision}")
    print("-" * 60)

    # --- STEP 2: Call the right specialist agent ---
    if decision == "ORDER":
        print("Handing off to: Order Specialist Agent")
        answer = run_specialist_agent(
            agent_graph=order_agent,
            system_prompt="""You are a Swiggy Order Specialist.
            You only handle order status and delivery tracking queries.
            Always check order status first, then provide delivery
            partner info if customer needs more details.""",
            query=customer_query
        )
    elif decision == "REFUND":
        print("Handing off to: Refund Specialist Agent")
        answer = run_specialist_agent(
            agent_graph=refund_agent,
            system_prompt="""You are a Swiggy Refund Specialist.
            You only handle refund and compensation queries.
            Always check if a refund already exists before creating
            a new one. Be empathetic and clear about refund timelines.""",
            query=customer_query
        )
    else:  # ESCALATION
        print("Handing off to: Escalation Specialist Agent")
        answer = run_specialist_agent(
            agent_graph=escalation_agent,
            system_prompt="""You are a Swiggy Escalation Specialist.
            You handle complex complaints that cannot be resolved by
            other agents. Always create a complaint ticket first,
            then escalate to human if needed.""",
            query=customer_query
        )

    print("-" * 60)
    print(f"Final answer: {answer}")
    print()
    return answer

# --- TEST ALL THREE ROUTES ---
print("🔵 TEST 1: Order query")
run_multi_agent_system("Where is my order ORD001? It is taking too long.")

print("🟢 TEST 2: Refund query")
run_multi_agent_system("I received wrong item in order ORD002. I want full refund.")

print("🔴 TEST 3: Escalation query")
run_multi_agent_system("The delivery partner for ORD003 was very rude and threatened me.")

## Summary

### What we built:
A complete Swiggy multi agent system with four agents:
- Router Agent — classifies query, routes to right specialist
- Order Agent — handles tracking with order tools only
- Refund Agent — handles refunds with refund tools only
- Escalation Agent — handles complaints with escalation tools only

### Key concepts:
- Specialist agents — each agent has one domain, one set of tools
- Router pattern — one agent receives all queries, routes to specialists
- build_specialist_agent() — DRY reusable function, built 3 agents in 15 lines
- Safety check on router output — always validate LLM decisions in code
- Separation of concerns — each agent is independently testable

### Why multi agent over single agent:
| Single Agent | Multi Agent |
|---|---|
| One agent does everything | Each agent is a specialist |
| Hard to debug | Easy to isolate issues |
| Hard to improve one part | Improve agents independently |
| All tools available always | Only relevant tools per agent |
| Does not scale | Scales horizontally |

### AWS Equivalent:
| This session | AWS |
|---|---|
| Router Agent | Amazon EventBridge rules |
| Order Agent | Lambda function — order domain |
| Refund Agent | Lambda function — refund domain |
| Escalation Agent | Lambda function — escalation domain |
| Full pipeline | Amazon Bedrock multi agent collaboration |
| build_specialist_agent() | AWS Lambda layers (reusable code) |

### Module 5 complete:
Session 1 — Tool Use and Function Calling
Session 2 — ReAct Agent Loop from scratch
Session 3 — Agents with LangGraph
Session 4 — Agent Memory (short term + long term)
Session 5 — Multi Agent Systems